# SalesLuv 딜 승패 모델 비교

이 노트북은 Salvirt B2B 영업 데이터로 다섯 모델을 같은 조건에서 비교합니다.

- 입력: 22개 범주형 컬럼
- 정답: `Lost=0`, `Won=1`
- 완전히 같은 23개 값의 중복 행 제거: 448건 → 365건
- 원본의 `Unknown`은 범주로 유지하고 합성 `Unknown`은 만들지 않음
- 분할: Train 70% / Test 30%, `random_state=1`, 층화 추출
- 선택 기준: Train 내부 5-Fold CV Brier Score
- Test는 최종 성능 확인에만 사용

## 0. 환경과 경로

저장소 루트에서 다음 명령으로 전용 환경의 JupyterLab을 실행합니다.

```bash
uv run --project backend/notebooks --locked jupyter lab \
  backend/notebooks/deal_model_phase1.ipynb
```

원본 CSV가 기본 경로(`/private/tmp/Salvirt_B2B_ML_dataset_HF.csv`)와 다르면 `SALESLUV_B2B_DATA_PATH` 환경변수로 지정합니다.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import warnings
from collections import Counter
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from tabicl import TabICLClassifier

warnings.filterwarnings(action="ignore")

MODEL_VERSION = "deal-model-comparison-v1"
SOURCE_SHA256 = "8dee635b95bdcb00896b654efe62fc20177090081c81ef5224e8641ba31c3061"
RANDOM_STATE = 1

ALL_COLUMNS = (
    "Product",
    "Seller",
    "Authority",
    "Comp_size",
    "Competitors",
    "Purch_dept",
    "Partnership",
    "Budgt_alloc",
    "Forml_tend",
    "RFI",
    "RFP",
    "Growth",
    "Posit_statm",
    "Source",
    "Client",
    "Scope",
    "Strat_deal",
    "Cross_sale",
    "Up_sale",
    "Deal_type",
    "Needs_def",
    "Att_t_client",
    "Status",
)
FEATURE_NAMES = ALL_COLUMNS[:-1]

current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    BACKEND_DIR = current_dir.parent
elif current_dir.name == "backend":
    BACKEND_DIR = current_dir
elif (current_dir / "backend" / "pipeline").is_dir():
    BACKEND_DIR = current_dir / "backend"
else:
    raise RuntimeError("backend directory not found")

DATA_PATH = Path(
    os.environ.get(
        "SALESLUV_B2B_DATA_PATH",
        "/private/tmp/Salvirt_B2B_ML_dataset_HF.csv",
    )
).expanduser()
ARTIFACT_PATH = BACKEND_DIR / "pipeline" / "artifacts" / f"{MODEL_VERSION}.pkl"

print(f"데이터: {DATA_PATH}")
print(f"모델 저장 경로: {ARTIFACT_PATH}")

## 1. 데이터 준비

In [ ]:
# 다른 버전의 CSV를 잘못 학습하지 않도록 원본 파일의 해시를 확인합니다.
source_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert source_sha256 == SOURCE_SHA256, f"unexpected source: {source_sha256}"

# 원본은 세미콜론으로 컬럼을 구분합니다. 모든 문자열의 앞뒤 공백을 정리합니다.
data = pd.read_csv(DATA_PATH, sep=";", encoding="utf-8-sig", dtype=str)
data = data.apply(lambda column: column.str.strip())
assert tuple(data.columns) == ALL_COLUMNS
assert not data.isna().any().any()
assert not data.eq("").any().any()

raw_rows = len(data)
data = data.drop_duplicates().reset_index(drop=True)
deduplicated_rows = len(data)

assert raw_rows == 448
assert deduplicated_rows == 365
assert data["Status"].value_counts().to_dict() == {"Lost": 192, "Won": 173}

print(f"원본 행: {raw_rows}")
print(f"중복 제거 후: {deduplicated_rows}")
print(f"제거된 중복: {raw_rows - deduplicated_rows}")
display(data.head(10))

In [ ]:
# X에는 22개 입력 컬럼을, y에는 Lost=0/Won=1 결과를 저장합니다.
target = "Status"
X = data.drop(columns=target).astype(str)
y = data[target].map({"Lost": 0, "Won": 1}).astype(int)

# Test는 파라미터 탐색에 사용하지 않고 마지막 평가에만 사용합니다.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.7,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {len(X_train)}, {dict(sorted(Counter(y_train).items()))}")
print(f"Test: {len(X_test)}, {dict(sorted(Counter(y_test).items()))}")

## 2. Dummy 기준선

모든 행에 Train의 평균 승률만 반환합니다. 실제 모델은 최소한 이 Brier보다 좋아야 합니다.

In [ ]:
model_dummy = DummyClassifier(strategy="prior")
cv_scores_dummy = cross_val_score(
    model_dummy,
    X_train,
    y_train,
    cv=5,
    scoring="neg_brier_score",
)
model_dummy.fit(X_train, y_train)

dummy_won_index = list(model_dummy.classes_).index(1)
probabilities_dummy = model_dummy.predict_proba(X_test)[:, dummy_won_index]
test_metrics_dummy = {
    "brier": float(brier_score_loss(y_test, probabilities_dummy)),
    "accuracy": float(accuracy_score(y_test, model_dummy.predict(X_test))),
    "auc": float(roc_auc_score(y_test, probabilities_dummy)),
    "logloss": float(log_loss(y_test, probabilities_dummy, labels=[0, 1])),
}

print(f"Dummy CV Brier: {-cv_scores_dummy.mean():.6f}")
print(f"Dummy Test: {test_metrics_dummy}")

## 3. LogisticRegression

`C`가 작을수록 L2 규제가 강해집니다.

In [ ]:
model_logistic = Pipeline(
    [
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
        (
            "classifier",
            LogisticRegression(
                l1_ratio=0,
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

cv_scores_logistic = cross_val_score(
    model_logistic,
    X_train,
    y_train,
    cv=5,
    scoring="neg_brier_score",
    n_jobs=-1,
)

param_logistic = {"classifier__C": [0.01, 0.1, 1.0, 10.0]}
search_logistic = GridSearchCV(
    model_logistic,
    param_logistic,
    cv=5,
    scoring="neg_brier_score",
    n_jobs=-1,
    error_score="raise",
)
search_logistic.fit(X_train, y_train)

y_pred_logistic = search_logistic.predict(X_test)
won_index_logistic = list(search_logistic.classes_).index(1)
probabilities_logistic = search_logistic.predict_proba(X_test)[:, won_index_logistic]
test_metrics_logistic = {
    "brier": float(brier_score_loss(y_test, probabilities_logistic)),
    "accuracy": float(accuracy_score(y_test, y_pred_logistic)),
    "auc": float(roc_auc_score(y_test, probabilities_logistic)),
    "logloss": float(log_loss(y_test, probabilities_logistic, labels=[0, 1])),
}

print(f"기본 CV Brier: {-cv_scores_logistic.mean():.6f}")
print("후보별 CV Brier:", -search_logistic.cv_results_["mean_test_score"])
print(f"최적 파라미터: {search_logistic.best_params_}")
print(f"최적 CV Brier: {-search_logistic.best_score_:.6f}")
print(f"Test: {test_metrics_logistic}")

## 4. MultinomialNB

`alpha`가 클수록 범주별 빈도 차이를 더 부드럽게 만듭니다. sigmoid 방식으로 확률을 보정합니다.

In [ ]:
model_nb = CalibratedClassifierCV(
    estimator=Pipeline(
        [
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ("classifier", MultinomialNB(alpha=1.0, fit_prior=True)),
        ]
    ),
    method="sigmoid",
    cv=5,
)

cv_scores_nb = cross_val_score(
    model_nb,
    X_train,
    y_train,
    cv=5,
    scoring="neg_brier_score",
    n_jobs=-1,
)

param_nb = {"estimator__classifier__alpha": [0.3, 0.6, 1.0, 2.0, 3.0]}
search_nb = GridSearchCV(
    model_nb,
    param_nb,
    cv=5,
    scoring="neg_brier_score",
    n_jobs=-1,
    error_score="raise",
)
search_nb.fit(X_train, y_train)

y_pred_nb = search_nb.predict(X_test)
won_index_nb = list(search_nb.classes_).index(1)
probabilities_nb = search_nb.predict_proba(X_test)[:, won_index_nb]
test_metrics_nb = {
    "brier": float(brier_score_loss(y_test, probabilities_nb)),
    "accuracy": float(accuracy_score(y_test, y_pred_nb)),
    "auc": float(roc_auc_score(y_test, probabilities_nb)),
    "logloss": float(log_loss(y_test, probabilities_nb, labels=[0, 1])),
}

print(f"기본 CV Brier: {-cv_scores_nb.mean():.6f}")
print("후보별 CV Brier:", -search_nb.cv_results_["mean_test_score"])
print(f"최적 파라미터: {search_nb.best_params_}")
print(f"최적 CV Brier: {-search_nb.best_score_:.6f}")
print(f"Test: {test_metrics_nb}")

## 5. ExtraTrees

트리 개수·깊이·잎의 최소 행 수·한 트리에서 사용할 입력 비율을 함께 탐색합니다.

In [ ]:
model_extratrees = Pipeline(
    [
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
        (
            "classifier",
            ExtraTreesClassifier(
                n_estimators=300,
                random_state=RANDOM_STATE,
                n_jobs=1,
            ),
        ),
    ]
)

cv_scores_extratrees = cross_val_score(
    model_extratrees,
    X_train,
    y_train,
    cv=5,
    scoring="neg_brier_score",
    n_jobs=-1,
)

param_extratrees = {
    "classifier__n_estimators": [300, 600],
    "classifier__max_depth": [None, 8, 16],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", 0.5],
}
search_extratrees = GridSearchCV(
    model_extratrees,
    param_extratrees,
    cv=5,
    scoring="neg_brier_score",
    n_jobs=-1,
    error_score="raise",
)
search_extratrees.fit(X_train, y_train)

y_pred_extratrees = search_extratrees.predict(X_test)
won_index_extratrees = list(search_extratrees.classes_).index(1)
probabilities_extratrees = search_extratrees.predict_proba(X_test)[:, won_index_extratrees]
test_metrics_extratrees = {
    "brier": float(brier_score_loss(y_test, probabilities_extratrees)),
    "accuracy": float(accuracy_score(y_test, y_pred_extratrees)),
    "auc": float(roc_auc_score(y_test, probabilities_extratrees)),
    "logloss": float(log_loss(y_test, probabilities_extratrees, labels=[0, 1])),
}

print(f"기본 CV Brier: {-cv_scores_extratrees.mean():.6f}")
print("후보별 CV Brier:", -search_extratrees.cv_results_["mean_test_score"])
print(f"최적 파라미터: {search_extratrees.best_params_}")
print(f"최적 CV Brier: {-search_extratrees.best_score_:.6f}")
print(f"Test: {test_metrics_extratrees}")

## 6. CatBoost

문자열 범주를 직접 처리합니다. 기존 탐색에서 의미가 있었던 세 설정만 비교합니다.

In [ ]:
model_catboost = CatBoostClassifier(
    iterations=400,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3.0,
    random_strength=1.0,
    loss_function="Logloss",
    auto_class_weights="Balanced",
    bootstrap_type="MVS",
    subsample=0.8,
    one_hot_max_size=2,
    verbose=False,
    allow_writing_files=False,
    random_seed=RANDOM_STATE,
    thread_count=1,
)
fit_params_catboost = {"cat_features": list(FEATURE_NAMES)}

cv_scores_catboost = cross_val_score(
    model_catboost,
    X_train,
    y_train,
    cv=5,
    scoring="neg_brier_score",
    n_jobs=-1,
    params=fit_params_catboost,
)

param_catboost = [
    {
        "depth": [6],
        "iterations": [400],
        "learning_rate": [0.03],
        "l2_leaf_reg": [3.0],
        "random_strength": [1.0],
    },
    {
        "depth": [4],
        "iterations": [400],
        "learning_rate": [0.03],
        "l2_leaf_reg": [3.0],
        "random_strength": [1.0],
    },
    {
        "depth": [4],
        "iterations": [500],
        "learning_rate": [0.025],
        "l2_leaf_reg": [10.0],
        "random_strength": [2.0],
    },
]
search_catboost = GridSearchCV(
    model_catboost,
    param_catboost,
    cv=5,
    scoring="neg_brier_score",
    n_jobs=-1,
    error_score="raise",
)
search_catboost.fit(X_train, y_train, **fit_params_catboost)

y_pred_catboost = search_catboost.predict(X_test)
won_index_catboost = list(search_catboost.classes_).index(1)
probabilities_catboost = search_catboost.predict_proba(X_test)[:, won_index_catboost]
test_metrics_catboost = {
    "brier": float(brier_score_loss(y_test, probabilities_catboost)),
    "accuracy": float(accuracy_score(y_test, y_pred_catboost)),
    "auc": float(roc_auc_score(y_test, probabilities_catboost)),
    "logloss": float(log_loss(y_test, probabilities_catboost, labels=[0, 1])),
}

print(f"기본 CV Brier: {-cv_scores_catboost.mean():.6f}")
print("후보별 CV Brier:", -search_catboost.cv_results_["mean_test_score"])
print(f"최적 파라미터: {search_catboost.best_params_}")
print(f"최적 CV Brier: {-search_catboost.best_score_:.6f}")
print(f"Test: {test_metrics_catboost}")

## 7. TabICL

사전학습된 표형 파운데이션 모델입니다. 첫 실행에는 공식 체크포인트를 내려받습니다.
MPS·CUDA 장치를 중복 사용하지 않도록 Grid Search는 순차 실행합니다.

In [ ]:
tabicl_device = "mps" if torch.backends.mps.is_available() else None
model_tabicl = TabICLClassifier(
    n_estimators=8,
    batch_size=8,
    kv_cache=False,
    allow_auto_download=True,
    device=tabicl_device,
    use_fa3="auto",
    offload_mode="auto",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=False,
)

cv_scores_tabicl = cross_val_score(
    model_tabicl,
    X_train,
    y_train,
    cv=5,
    scoring="neg_brier_score",
    n_jobs=1,
)

param_tabicl = [
    {
        "n_estimators": [8],
        "norm_methods": [None],
        "feat_shuffle_method": ["latin"],
        "average_logits": [True],
        "softmax_temperature": [0.9, 1.1, 1.3],
    },
    {
        "n_estimators": [8],
        "norm_methods": ["quantile"],
        "feat_shuffle_method": ["latin"],
        "average_logits": [True],
        "softmax_temperature": [0.9],
    },
]
search_tabicl = GridSearchCV(
    model_tabicl,
    param_tabicl,
    cv=5,
    scoring="neg_brier_score",
    n_jobs=1,
    error_score="raise",
)
search_tabicl.fit(X_train, y_train)

y_pred_tabicl = search_tabicl.predict(X_test)
won_index_tabicl = list(search_tabicl.classes_).index(1)
probabilities_tabicl = search_tabicl.predict_proba(X_test)[:, won_index_tabicl]
test_metrics_tabicl = {
    "brier": float(brier_score_loss(y_test, probabilities_tabicl)),
    "accuracy": float(accuracy_score(y_test, y_pred_tabicl)),
    "auc": float(roc_auc_score(y_test, probabilities_tabicl)),
    "logloss": float(log_loss(y_test, probabilities_tabicl, labels=[0, 1])),
}

print(f"기본 CV Brier: {-cv_scores_tabicl.mean():.6f}")
print("후보별 CV Brier:", -search_tabicl.cv_results_["mean_test_score"])
print(f"최적 파라미터: {search_tabicl.best_params_}")
print(f"최적 CV Brier: {-search_tabicl.best_score_:.6f}")
print(f"Test: {test_metrics_tabicl}")

## 8. 모델 비교

최종 모델은 Test가 아니라 Train 내부 최적 CV Brier가 가장 낮은 모델로 선택합니다.

In [ ]:
trained_models = {
    "LogisticRegression": search_logistic,
    "MultinomialNB": search_nb,
    "ExtraTrees": search_extratrees,
    "CatBoost": search_catboost,
    "TabICL": search_tabicl,
}
test_metrics_by_model = {
    "LogisticRegression": test_metrics_logistic,
    "MultinomialNB": test_metrics_nb,
    "ExtraTrees": test_metrics_extratrees,
    "CatBoost": test_metrics_catboost,
    "TabICL": test_metrics_tabicl,
}

comparison = pd.DataFrame(
    [
        {
            "model": "Dummy",
            "cv_brier": float(-cv_scores_dummy.mean()),
            "test_brier": test_metrics_dummy["brier"],
            "test_auc": test_metrics_dummy["auc"],
            "test_accuracy": test_metrics_dummy["accuracy"],
            "best_params": {},
        },
        *[
            {
                "model": name,
                "cv_brier": float(-search.best_score_),
                "test_brier": test_metrics_by_model[name]["brier"],
                "test_auc": test_metrics_by_model[name]["auc"],
                "test_accuracy": test_metrics_by_model[name]["accuracy"],
                "best_params": search.best_params_,
            }
            for name, search in trained_models.items()
        ],
    ]
).sort_values("cv_brier", ignore_index=True)

display(comparison)

selected_model_name = comparison.loc[comparison["model"] != "Dummy", "model"].iloc[0]
selected_model = trained_models[selected_model_name]
print(f"최종 선택 모델: {selected_model_name}")
print(f"최종 선택 모델 CV Brier: {-selected_model.best_score_:.6f}")

## 9. 최종 모델 저장

위의 비교표를 확인한 뒤 이 셀을 실행합니다. TabICL이 선택되면 외부 체크포인트 없이
불러올 수 있도록 모델 가중치와 학습 문맥을 함께 저장합니다. 생성 파일은 Git에서 제외됩니다.

In [ ]:
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
expected_probabilities = selected_model.predict_proba(X_test.iloc[:16])

if selected_model_name == "TabICL":
    artifact_format = "tabicl-native-with-weights"
    selected_model.best_estimator_.save(
        ARTIFACT_PATH,
        save_model_weights=True,
        save_training_data=True,
        save_kv_cache=False,
    )
    restored = TabICLClassifier.load(ARTIFACT_PATH, device=tabicl_device)
else:
    artifact_format = "joblib"
    joblib.dump(selected_model.best_estimator_, ARTIFACT_PATH, compress=3)
    restored = joblib.load(ARTIFACT_PATH)

np.testing.assert_allclose(
    restored.predict_proba(X_test.iloc[:16]),
    expected_probabilities,
    rtol=1e-6,
    atol=1e-7,
)


def search_summary(search: GridSearchCV) -> list[dict[str, object]]:
    return [
        {
            "params": params,
            "mean_cv_brier": float(-mean_score),
            "std_cv_brier": float(std_score),
        }
        for params, mean_score, std_score in zip(
            search.cv_results_["params"],
            search.cv_results_["mean_test_score"],
            search.cv_results_["std_test_score"],
            strict=True,
        )
    ]


artifact_sha256 = hashlib.sha256(ARTIFACT_PATH.read_bytes()).hexdigest()
metadata = {
    "model_version": MODEL_VERSION,
    "selected_model": selected_model_name,
    "selected_best_params": selected_model.best_params_,
    "selected_cv_brier": float(-selected_model.best_score_),
    "artifact": str(ARTIFACT_PATH),
    "artifact_format": artifact_format,
    "artifact_sha256": artifact_sha256,
    "trained_at_utc": datetime.now(UTC).isoformat(),
    "feature_names": list(FEATURE_NAMES),
    "target": {"Lost": 0, "Won": 1},
    "data": {
        "source_sha256": source_sha256,
        "raw_rows": raw_rows,
        "deduplicated_rows": deduplicated_rows,
        "removed_duplicates": raw_rows - deduplicated_rows,
        "synthetic_unknowns": False,
    },
    "split": {
        "train_size": 0.7,
        "test_size": 0.3,
        "random_state": RANDOM_STATE,
        "stratified": True,
        "training_rows": len(X_train),
        "test_rows": len(X_test),
    },
    "comparison": comparison.to_dict(orient="records"),
    "searches": {name: search_summary(search) for name, search in trained_models.items()},
    "versions": {
        "python": platform.python_version(),
        "catboost": version("catboost"),
        "joblib": version("joblib"),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "tabicl": version("tabicl"),
        "torch": torch.__version__,
    },
    "self_check": "reloaded_predictions_match",
}
metadata_path = ARTIFACT_PATH.with_suffix(".json")
metadata_path.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2, default=str) + "\n",
    encoding="utf-8",
)

print(f"모델: {ARTIFACT_PATH}")
print(f"메타데이터: {metadata_path}")
print(f"SHA256: {artifact_sha256}")